# Assignment 10

**Student roll number:** 1024170115

## Solution 1 — Labelled query corpus (50 examples, 5 intents)
The five intents are `glucose`, `medication`, `food_activity`, `appointments`, and `emergency`.

In [ ]:
queries = {
"glucose": ["How do I check my blood sugar", "When should I measure glucose", "What does a glucose reading mean", "How often should I log my sugar", "Can I track my fasting glucose", "My meter says 140 what does that mean", "How do I record glucose results", "What is a blood sugar target", "Why should I check after meals", "How can I use a glucose diary"],
"medication": ["How should I store insulin", "What if I missed a diabetes dose", "Can I take insulin before eating", "How do I remember my medication", "What supplies are used for insulin", "Where should insulin pens be kept", "Can I change my dose myself", "What should I ask about side effects", "How do I dispose of needles safely", "Does my medicine need refrigeration"],
"food_activity": ["What foods can help manage diabetes", "How do carbohydrates affect glucose", "Can walking help blood sugar", "Suggest a balanced meal idea", "How can I read food labels", "Is regular exercise helpful", "What is a portion size", "Should I drink water during exercise", "How can I plan snacks", "What meals include vegetables and protein"],
"appointments": ["How should I prepare for a checkup", "Which questions should I ask my doctor", "What tests are used for diabetes follow up", "How often should I schedule a review", "What information should I bring to clinic", "Can I discuss my glucose log at the visit", "What should I ask my diabetes educator", "How do I prepare a medicine list", "What happens during an eye check", "Should I note symptoms before my appointment"],
"emergency": ["I am unconscious after taking insulin", "I have severe trouble breathing", "My blood sugar is extremely low and I am confused", "I have chest pain and feel faint", "I cannot wake my family member", "I am having a seizure", "My breathing is labored and I feel weak", "I am confused and cannot swallow", "There is severe bleeding", "I may pass out and cannot get help"]
}
records = [{"query": q, "intent": intent} for intent, items in queries.items() for q in items]
len(records), pd.DataFrame(records).groupby("intent").size()

In [ ]:
import re, pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Preprocessing: lowercase, tokenize, remove stopwords, lemmatize.
import nltk
nltk.download("stopwords", quiet=True); nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words("english")); lemmatizer = WordNetLemmatizer()
def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    return " ".join(lemmatizer.lemmatize(t) for t in tokens if t not in stop_words)
X = [preprocess(r["query"]) for r in records]
y = [r["intent"] for r in records]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2))), ("classifier", LinearSVC())])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision, recall, F1 (macro):", precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)[:3])
labels = sorted(set(y))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=labels), display_labels=labels).plot(xticks_rotation=45); plt.tight_layout(); plt.show()

## Solution 2 — Hierarchical taxonomy
- Diabetes care
  - glucose monitoring
  - medication use
- Daily living
  - food planning
  - physical activity
- Clinical care
  - appointments
  - tests and follow-up
- Safety
  - urgent symptoms
  - emergency response

## Solution 3 — Domain concepts and relationships
Concepts: person, diabetes, glucose, glucose meter, insulin, meal, carbohydrate, exercise, clinician, appointment, symptom, emergency service.

Relationships: person manages diabetes; diabetes affects glucose; meter measures glucose; insulin can lower glucose; meal contains carbohydrate; carbohydrate can change glucose; exercise can affect glucose; person records glucose; clinician reviews glucose records; person schedules appointment; symptom may require clinician review; emergency symptom requires emergency service.

## Solution 4 — Knowledge base by intent

In [ ]:
knowledge_base = {
"glucose": "Use your care team’s instructions for when and how to check glucose. Record readings with date, time, meals, activity, and symptoms. Ask your clinician what range is appropriate for you.",
"medication": "Follow the medication plan supplied by your clinician or pharmacist. Do not change a dose based on this assistant. Ask a qualified professional what to do after a missed dose and how to store or dispose of supplies.",
"food_activity": "A balanced eating pattern and regular physical activity can support diabetes management. Individual needs differ; discuss meal planning and exercise safety with your care team, especially if you use glucose-lowering medication.",
"appointments": "Bring a current medicine list, glucose records, questions, and notes about symptoms to your appointment. Your clinician can advise on tests and follow-up timing.",
"emergency": "This may be urgent. Contact your local emergency service now or ask someone nearby to help. Do not wait for an online assistant. If the person is unconscious or unable to swallow, do not give food or drink by mouth."
}


## Solution 5 — Rules and integrated assistant
Rules: (1) emergency language always escalates; (2) low classifier confidence requests clarification; (3) unsupported topics are declined; (4) medication changes are directed to a clinician/pharmacist; (5) informational answers include general-scope framing; (6) intent matching uses confidence from a calibrated text classifier.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
# Calibrate scores so the low-confidence rule has a probability-like value.
calibrated = CalibratedClassifierCV(LinearSVC(), method="sigmoid", cv=3)
confidence_model = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2))), ("classifier", calibrated)])
confidence_model.fit(X_train, y_train)

emergency_terms = ["unconscious", "seizure", "cannot breathe", "trouble breathing", "chest pain", "cannot swallow", "pass out", "extremely low"]
medication_change_terms = ["change my dose", "increase my dose", "stop taking", "double my dose"]
def assistant(query):
    q = query.lower()
    if any(term in q for term in emergency_terms):
        return knowledge_base["emergency"]
    if any(term in q for term in medication_change_terms):
        return "Do not change or stop a diabetes medicine based on this assistant. Contact your clinician or pharmacist for advice. If you feel severely unwell, contact emergency services."
    if any(term in q for term in ["cancer", "pregnancy", "broken bone"]):
        return "I do not have reliable information for that topic. Please contact a qualified healthcare professional."
    clean = preprocess(query)
    probabilities = confidence_model.predict_proba([clean])[0]
    best_i = int(probabilities.argmax()); intent = confidence_model.classes_[best_i]; confidence = probabilities[best_i]
    if confidence < 0.30:
        return "I am not confident which diabetes topic you mean. Could you rephrase or choose glucose, medication, food/activity, or appointments?"
    return knowledge_base[intent] + f"\n\n(Detected topic: {intent}; confidence: {confidence:.2f})"

for q in ["How do I record fasting glucose?", "I forgot my insulin dose", "Can walking help blood sugar?", "What should I bring to a checkup?", "I am unconscious after taking insulin", "Who won the cricket match?"]:
    print("Q:", q, "\nA:", assistant(q), "\n")

## Solution 6 — Tests required by the assignment
The queries above cover each intent, a reworded version, an unsupported topic, an ambiguous low-confidence query, and an emergency. Add one more test query of your own and inspect whether the answer is appropriate. The confidence threshold is a simple coursework rule, not a clinically validated safety measure.

In [ ]:
test_queries = [
    "Can you explain my morning sugar reading?",
    "I feel dizzy, confused, and might pass out",
    "Tell me how to bake a cake",
    "What is the treatment for cancer?",
    "What is the treatment for cancer?",
    "What is the treatment for cancer?",
    "qzxv plmokn",
    "qzxv plmokn",
    "qzxv plmokn",
    "What should I ask at my next diabetes review?"
]
for q in test_queries:
    print(f"Q: {q}\nA: {assistant(q)}\n")

## Solution 7 — Interactive interface for Google Colab
The loop accepts multiple queries. It works in Colab without extra interface libraries.

In [ ]:
print("Diabetes Care Assistant (general information only). Type quit to end.")
while True:
    user_query = input("You: ").strip()
    if user_query.lower() in {"quit", "exit"}:
        print("Assistant: Take care.")
        break
    print("Assistant:", assistant(user_query))